In [ ]:
# Shared setup and reusable notebook helpers
import importlib
import os
import subprocess
import sys
from pathlib import Path
import numpy as np
import illustris_python as il

root = Path("/public/home/zju_visitor/LiuYuanhao/SAM_project/bymyself")
data_dir = root / "data"
basePath = "/public/share/chenhouzun/TNG50-1/output/"

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

env = os.environ.copy()
env["PYTHONPATH"] = str(root) + os.pathsep + env.get("PYTHONPATH", "")

def run_script(script, *args):
    subprocess.run([sys.executable, str(script), *map(str, args)], cwd=root, env=env, check=True)

def run_snapshots(script, output_template, label, snap_list=snaps):
    for snap in snap_list:
        output_file = data_dir / output_template.format(snap=snap)
        if output_file.is_file():
            print(f"{label} file for snap {snap} already exists, skip.")
            continue
        print(f"Running {label} calculation for snap {snap}")
        run_script(script, snap)
    print(f"All {label} calculations finished.")

def run_plot(script, finished_message):
    run_script(script)
    print(finished_message)

In [ ]:
# Plot the TNG hot-gas profiles for snaps 40, 67, and 99
properties_plot_script = root / "plotting_scripts" / "plot_properties_tng_Lgal.py"
run_plot(properties_plot_script, "Saved the multi-redshift hot-gas profile figure.")

In [ ]:
# Compare Lgal analytic and TNG-measured cooling radii for snaps 99, 67, and 40
cooling_radius_plot_script = root / "plotting_scripts" / "plot_Lgal_tng_cooling_radius_mass_trend_z.py"
cooling_radius_inputs = [data_dir / "all_selected_trees.hdf5"]
for snap in (99, 67, 40):
    cooling_radius_inputs.extend((data_dir / f"cooling_radius_{snap}.hdf5", data_dir / f"real_cooling_radius_{snap}.hdf5"))
missing_inputs = [path for path in cooling_radius_inputs if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError(f"Missing cooling-radius plot inputs: {missing_inputs}")
run_plot(cooling_radius_plot_script, "Saved the Lgal versus TNG cooling-radius trend plot.")